# 📑Tutorial-10: Simulated Side-Channel Trace Generation

**🧪 Simulated trace generation with traceset.sim**

When no real oscilloscope or target device is available, the NUSCAR `traceset.sim` module generates "attackable" simulated side-channel traces directly, which is handy for algorithm validation, tutorial demos and unit tests.

The idea is simple: leakage values (such as the Hamming weight of intermediate values) are added onto a few sample positions, Gaussian noise is added on top, and the result is still an ordinary `ContainerMemory` that plugs straight into the existing `task`, `viewer` and `distinguisher` workflows.

The module provides four functions:
- ✅ **simulate_traces**: general interface that takes a leakage matrix directly
- ✅ **simulate_aes_traces**: AES-128 specific, generates the plaintext/key automatically and computes the leakage of the first-round intermediate values
- ✅ **simulate_traces_to_disk** / **simulate_aes_traces_to_disk**: for datasets too large to fit in memory, writing memmap `.npy` files while generating; the random numbers of each trace are derived independently from `(seed, trace index)`, so generation is reproducible and safe to parallelise

In [1]:
import nuscar
import numpy as np

### General simulation: simulate_traces

Provide your own leakage matrix (one column per leak point); `simulate_traces` adds these values onto the selected sample positions, adds Gaussian noise, and returns an ordinary `ContainerMemory`.

In [3]:
rng = np.random.default_rng(0)
leak = rng.integers(0, 9, size=(2000, 3)).astype(np.float64)  # 3 leak points, values imitate the Hamming weight of a byte (0~8)
ctn = nuscar.traceset.simulate_traces(leak, nb_samples=200, noise=1.0, seed=1)
ctn

Trace Count,Sample Points,Meta Info
2000,200,[]


In [4]:
ctn.poi  # positions of the 3 leak points along the sample axis (evenly spread over [0.1, 0.9] * nb_samples by default)

array([ 20, 100, 180])

In [5]:
ctn.view([0, 1, 2])  # display a few simulated traces (returns an interactive ipywidgets control)

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

### AES simulation: simulate_aes_traces

`simulate_aes_traces` ships with a built-in AES-128 simulation: it generates random plaintexts and uses the Hamming weight of the first-round S-box output (or the AddRoundKey output) as the leakage, embedded on the sample position of each byte. The generated traces come with `plaintext`/`key` metadata, so the attack helpers in `nuscar.ciphers.aes` can analyse them directly.

In [6]:
key = np.frombuffer(bytes.fromhex("2b7e151628aed2a6abf7158809cf4f3c"), dtype='uint8')
ctn = nuscar.traceset.simulate_aes_traces(nb_traces=3000, nb_samples=200, key=key, noise=1.0, seed=7)
ctn

Trace Count,Sample Points,Meta Info
3000,200,"['plaintext', 'key']"


In [7]:
ctn.viewmeta()  # inspect the metadata: plaintext and key

No.,plaintext,key
0,"[139, 74, 229, 241, 169, 65, 6, 160, 149, 106, 38, 175, 188, 205, 175, 229]","[43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60]"
1,"[98, 249, 10, 148, 95, 86, 147, 198, 66, 39, 106, 213, 171, 45, 167, 57]","[43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60]"
2,"[69, 81, 55, 14, 153, 178, 215, 76, 52, 39, 250, 72, 215, 50, 161, 223]","[43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60]"
3,"[112, 172, 161, 233, 38, 17, 89, 1, 221, 6, 242, 127, 143, 6, 60, 210]","[43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60]"
4,"[94, 25, 166, 33, 249, 189, 12, 204, 33, 57, 124, 30, 199, 149, 202, 119]","[43, 126, 21, 22, 40, 174, 210, 166, 171, 247, 21, 136, 9, 207, 79, 60]"


In [8]:
ctn.poi  # leak position of each of the 16 bytes

array([ 20,  31,  41,  52,  63,  73,  84,  95, 105, 116, 127, 137, 148,
       159, 169, 180])

### Verifying the attack with a CPA distinguisher

Because the simulated traces leak the Hamming weight of the first-round S-box output, the `attack_first_sbox_hw` selection function can be reused directly to check whether the recovered key matches the `key` used for the simulation.

In [9]:
sf_attack_sbox = nuscar.ciphers.aes.attack_first_sbox_hw()
dist = nuscar.distinguisher.CPADistinguisher()
task = nuscar.task.DistinguisherTask(ctn, sf_attack_sbox, distinguisher=dist)
task.run()

  0%|          | 0/3000 [00:00<?, ?it/s]

2026-07-22 09:27:08,460 - nuscar - INFO - check selection func pass
2026-07-22 09:27:08,461 - nuscar - INFO - check memory pass
2026-07-22 09:27:08,462 - nuscar - INFO - task Dist Task: 3000 traces, 1 distinguisher, batch_size=25000, nb_samples=200
2026-07-22 09:27:08,609 - nuscar - INFO - Task completed, time taken: 0.1 seconds


In [10]:
task.show_candidate(correct_key=key)  # compare against the key used by the simulation to confirm the recovery

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
rank 1,43,126,21,22,40,174,210,166,171,247,21,136,9,207,79,60
rank 2,222,139,8,227,221,44,24,152,12,2,224,125,252,58,186,201
rank 3,20,217,224,41,23,91,39,83,148,200,137,183,54,114,247,160


### Writing large datasets to disk: simulate_aes_traces_to_disk

When `nb_traces * nb_samples` is too large to hold in memory at once, `simulate_aes_traces_to_disk` writes to disk while generating (memmap `.npy` files), and the result can be read on demand with `ContainerNPY.from_dir` (`mmap_mode='r'`).

The random numbers of every trace are derived independently with `np.random.default_rng((seed, trace index))`, so:
- the same `seed` produces exactly the same per-trace results no matter which `batch_size` or `n_jobs` is used;
- a single trace can be regenerated on its own for verification instead of rerunning the whole dataset;
- `n_jobs` parallelises generation safely (joblib's threading backend, used the same way as `Storer.update_sync`).

If the target directory already exists an error is raised by default; pass `overwrite=True` explicitly to replace it, which avoids deleting generated data by accident.

In [12]:
out_dir = nuscar.traceset.simulate_aes_traces_to_disk(
    'aes_sim_disk', nb_traces=5000, nb_samples=300, key=key,
    noise=1.0, seed=42, batch_size=1000, n_jobs=-1, overwrite=True
)
out_dir

  0%|          | 0/5000 [00:00<?, ?it/s]

2026-07-22 09:27:23,423 - nuscar - INFO - Wrote 5000 traces to aes_sim_disk


PosixPath('aes_sim_disk')

In [13]:
ctn_disk = nuscar.traceset.ContainerNPY.from_dir(out_dir, mmap_mode='r')  # read with mmap instead of loading everything into memory
ctn_disk

Trace Count,Sample Points,Meta Info
5000,300,"['key', 'plaintext']"


In [14]:
ctn_disk.poi

memmap([ 30,  46,  62,  78,  94, 110, 126, 142, 158, 174, 190, 206, 222,
        238, 254, 270])

The data on disk plugs straight into the existing attack workflow:

In [15]:
task2 = nuscar.task.DistinguisherTask(
    ctn_disk, sf_attack_sbox, distinguisher=nuscar.distinguisher.CPADistinguisher())
task2.run()
task2.show_candidate(correct_key=key)

  0%|          | 0/5000 [00:00<?, ?it/s]

2026-07-22 09:27:29,944 - nuscar - INFO - check selection func pass
2026-07-22 09:27:29,945 - nuscar - INFO - check memory pass
2026-07-22 09:27:29,945 - nuscar - INFO - task Dist Task: 5000 traces, 1 distinguisher, batch_size=25000, nb_samples=300
2026-07-22 09:27:30,229 - nuscar - INFO - Task completed, time taken: 0.3 seconds


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
rank 1,43,126,21,22,40,174,210,166,171,247,21,136,9,207,79,60
rank 2,222,139,224,227,221,91,237,83,94,2,224,66,252,58,186,201
rank 3,21,150,42,41,23,145,39,147,148,72,42,183,58,217,232,190


### Custom leakage: simulate_traces_to_disk

If your target is not the built-in AES, use the lower-level `simulate_traces_to_disk` and provide your own function `gen_leak(indices, seed)` that computes the leakage batch by batch.

In [18]:
def gen_leak(indices, seed):
    # each trace derives its own random numbers from (seed, trace index), which keeps generation reproducible and parallel-safe
    leak = np.empty((len(indices), 2), dtype=np.float64)
    for row, idx in enumerate(indices):
        rng = np.random.default_rng((seed, int(idx)))
        leak[row] = rng.integers(0, 9, size=2)  # 2 custom leak points
    return leak, {}  # no extra metadata

out_dir2 = nuscar.traceset.simulate_traces_to_disk(
    'custom_sim_disk', nb_traces=1000, nb_samples=100, gen_leak=gen_leak,
    nb_points=2, noise=0.5, seed=3, overwrite=True
)
ctn_custom = nuscar.traceset.ContainerNPY.from_dir(out_dir2)
ctn_custom

  0%|          | 0/1000 [00:00<?, ?it/s]

2026-07-22 09:27:44,697 - nuscar - INFO - Wrote 1000 traces to custom_sim_disk


Trace Count,Sample Points,Meta Info
1000,100,[]
